# 05 - Future Projections: Life Expectancy to 2100

This notebook visualizes life expectancy projections for Blue Zone countries and the world.

**Projection source:** Trend extrapolation from real historical data with logistic dampening
(the UN Population Division API was unavailable at collection time). Projections include
Medium, High, and Low scenarios with growing uncertainty bands.

**Caveat:** All projections are model outputs with inherent uncertainty. The actual future
may fall outside the projected ranges.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

PROJECT_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
    PROJECT_DIR = os.getcwd()
    if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
        PROJECT_DIR = os.path.dirname(PROJECT_DIR)

hist = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'historical', 'merged_historical_panel.csv'))
proj = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'projections', 'un_life_expectancy_projections.csv'))

BZ_INFO = {
    'JPN': {'name': 'Japan', 'color': '#E74C3C'},
    'ITA': {'name': 'Italy', 'color': '#2ECC71'},
    'GRC': {'name': 'Greece', 'color': '#9B59B6'},
    'CRI': {'name': 'Costa Rica', 'color': '#F39C12'},
    'USA': {'name': 'USA', 'color': '#3498DB'},
}

print(f'Historical: {hist["year"].min()}-{hist["year"].max()} ({len(hist):,} rows)')
print(f'Projections: {proj["year"].min()}-{proj["year"].max()} ({len(proj):,} rows)')
print(f'Projection source: {proj["projection_source"].iloc[0]}')

Historical: 1960-2023 (5,952 rows)
Projections: 2024-2100 (7,161 rows)
Projection source: trend_extrapolation_from_real_data


## 1. Blue Zone Countries: Historical + Projected Life Expectancy

In [2]:
fig, ax = plt.subplots(figsize=(16, 8))

for iso, info in BZ_INFO.items():
    # Historical
    h = hist[(hist['iso_code'] == iso) & (hist['life_expectancy'].notna())].sort_values('year')
    ax.plot(h['year'], h['life_expectancy'], '-', color=info['color'], linewidth=2,
            label=f'{info["name"]} (historical)')
    
    # Projections
    p = proj[proj['iso_code'] == iso].sort_values('year')
    if len(p) > 0:
        ax.plot(p['year'], p['le_medium'], '--', color=info['color'], linewidth=1.5, alpha=0.8)
        ax.fill_between(p['year'], p['le_low'], p['le_high'],
                       alpha=0.08, color=info['color'])

# Global average historical
global_hist = hist[hist['life_expectancy'].notna()].groupby('year')['life_expectancy'].mean()
ax.plot(global_hist.index, global_hist.values, 'k-', linewidth=1.5, alpha=0.4,
        label='Global avg (historical)')

# Global average projected
global_proj = proj.groupby('year')[['le_medium', 'le_low', 'le_high']].mean()
ax.plot(global_proj.index, global_proj['le_medium'], 'k--', linewidth=1.5, alpha=0.4,
        label='Global avg (projected)')

# Vertical line at transition
ax.axvline(x=2023, color='gray', linestyle=':', alpha=0.5)
ax.text(2024, ax.get_ylim()[0] + 2, 'Projections', fontsize=9, color='gray')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Life Expectancy (years)', fontsize=12)
ax.set_title('Blue Zone Countries: Life Expectancy 1960-2100 (Historical + Projections)', fontsize=14)
ax.legend(fontsize=9, loc='lower right', ncol=2)
ax.set_xlim(1960, 2100)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb05_full_timeline.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb05_full_timeline.png')

Saved: nb05_full_timeline.png


/tmp/ipykernel_1595051/2541363220.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Projected Values at Key Years

In [3]:
key_future_years = [2030, 2040, 2050, 2075, 2100]

for year in key_future_years:
    print(f'\n--- {year} Projections ---')
    for iso, info in BZ_INFO.items():
        p = proj[(proj['iso_code'] == iso) & (proj['year'] == year)]
        if len(p) > 0:
            r = p.iloc[0]
            print(f'  {info["name"]}: {r["le_medium"]:.1f} years '
                  f'(range: {r["le_low"]:.1f} - {r["le_high"]:.1f})')


--- 2030 Projections ---
  Japan: 84.2 years (range: 83.3 - 85.0)
  Italy: 83.8 years (range: 83.0 - 84.7)
  Greece: 81.7 years (range: 80.8 - 82.5)
  Costa Rica: 80.9 years (range: 80.0 - 81.7)
  USA: 78.4 years (range: 77.5 - 79.2)

--- 2040 Projections ---
  Japan: 84.3 years (range: 83.0 - 85.7)
  Italy: 84.0 years (range: 82.6 - 85.3)
  Greece: 81.8 years (range: 80.5 - 83.2)
  Costa Rica: 81.0 years (range: 79.6 - 82.3)
  USA: 78.4 years (range: 77.1 - 79.8)

--- 2050 Projections ---
  Japan: 84.5 years (range: 82.6 - 86.3)
  Italy: 84.2 years (range: 82.3 - 86.0)
  Greece: 82.0 years (range: 80.2 - 83.8)
  Costa Rica: 81.1 years (range: 79.2 - 82.9)
  USA: 78.4 years (range: 76.6 - 80.3)

--- 2075 Projections ---
  Japan: 84.9 years (range: 81.8 - 88.0)
  Italy: 84.6 years (range: 81.5 - 87.7)
  Greece: 82.4 years (range: 79.3 - 85.5)
  Costa Rica: 81.3 years (range: 78.2 - 84.4)
  USA: 78.5 years (range: 75.4 - 81.6)

--- 2100 Projections ---
  Japan: 85.3 years (range: 80.9 -

  Greece: 82.9 years (range: 78.5 - 87.2)
  Costa Rica: 81.5 years (range: 77.2 - 85.9)
  USA: 78.5 years (range: 74.2 - 82.9)


## 3. When Will the Global Average Reach Current Blue Zone Levels?

In [4]:
# Current BZ average
bz_current = hist[(hist['is_blue_zone'] == 1) & (hist['year'] >= 2020) &
                   (hist['life_expectancy'].notna())]['life_expectancy'].mean()
print(f'Current Blue Zone country average LE: {bz_current:.1f} years')

# When does global projected average reach this?
global_proj_med = proj.groupby('year')['le_medium'].mean()
crossover = global_proj_med[global_proj_med >= bz_current]
if len(crossover) > 0:
    print(f'Global average projected to reach {bz_current:.1f} years by: {crossover.index[0]}')
else:
    print(f'Global average not projected to reach {bz_current:.1f} years by 2100')

print(f'\nGlobal average projected LE in 2050: {global_proj_med.get(2050, "N/A"):.1f} years')
print(f'Global average projected LE in 2100: {global_proj_med.get(2100, "N/A"):.1f} years')

Current Blue Zone country average LE: 80.9 years
Global average not projected to reach 80.9 years by 2100

Global average projected LE in 2050: 77.2 years
Global average projected LE in 2100: 79.8 years


## 4. Projected Gap: Blue Zone vs Global

In [5]:
# Calculate projected BZ avg vs global avg
bz_proj = proj[proj['is_blue_zone'] == 1].groupby('year')['le_medium'].mean()
all_proj = proj.groupby('year')['le_medium'].mean()

proj_gap = bz_proj - all_proj

# Historical gap
bz_hist_avg = hist[(hist['is_blue_zone'] == 1) & (hist['life_expectancy'].notna())].groupby('year')['life_expectancy'].mean()
all_hist_avg = hist[hist['life_expectancy'].notna()].groupby('year')['life_expectancy'].mean()
hist_gap = bz_hist_avg - all_hist_avg

fig, ax = plt.subplots(figsize=(14, 6))

# Historical gap
ax.plot(hist_gap.index, hist_gap.values, 'b-', linewidth=2, label='Historical gap')
ax.fill_between(hist_gap.index, 0, hist_gap.values, alpha=0.15, color='blue')

# Projected gap
ax.plot(proj_gap.index, proj_gap.values, 'b--', linewidth=1.5, alpha=0.7, label='Projected gap')
ax.fill_between(proj_gap.index, 0, proj_gap.values, alpha=0.08, color='blue')

ax.axvline(x=2023, color='gray', linestyle=':', alpha=0.5)
ax.axhline(y=0, color='black', linewidth=0.5)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('BZ Country Avg - Global Avg (years)', fontsize=12)
ax.set_title('Blue Zone Advantage Over Global Average: Historical and Projected', fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb05_projected_gap.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb05_projected_gap.png')

Saved: nb05_projected_gap.png


/tmp/ipykernel_1595051/3798932064.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Uncertainty Fan Chart for Japan (Highest LE Blue Zone Country)

In [6]:
fig, ax = plt.subplots(figsize=(14, 7))

# Japan historical
jpn_hist = hist[(hist['iso_code'] == 'JPN') & (hist['life_expectancy'].notna())].sort_values('year')
ax.plot(jpn_hist['year'], jpn_hist['life_expectancy'], 'b-', linewidth=2.5, label='Historical')

# Japan projections
jpn_proj = proj[proj['iso_code'] == 'JPN'].sort_values('year')
ax.plot(jpn_proj['year'], jpn_proj['le_medium'], 'b--', linewidth=2, label='Medium projection')
ax.fill_between(jpn_proj['year'], jpn_proj['le_low'], jpn_proj['le_high'],
               alpha=0.15, color='blue', label='High/Low range')

ax.axvline(x=2023, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Life Expectancy (years)', fontsize=12)
ax.set_title('Japan Life Expectancy: Historical Data + Projections to 2100', fontsize=14)
ax.legend(fontsize=11)

# Key milestones
jpn_2050 = jpn_proj[jpn_proj['year'] == 2050]
if len(jpn_2050) > 0:
    v = jpn_2050.iloc[0]['le_medium']
    ax.annotate(f'2050: {v:.1f}', xy=(2050, v), xytext=(2055, v - 3),
               fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

jpn_2100 = jpn_proj[jpn_proj['year'] == 2100]
if len(jpn_2100) > 0:
    v = jpn_2100.iloc[0]['le_medium']
    ax.annotate(f'2100: {v:.1f}', xy=(2100, v), xytext=(2085, v - 3),
               fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb05_japan_fan_chart.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb05_japan_fan_chart.png')

Saved: nb05_japan_fan_chart.png


/tmp/ipykernel_1595051/2942832495.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Summary

In [7]:
print('PROJECTION ANALYSIS SUMMARY')
print('=' * 50)
print(f'\nProjection source: {proj["projection_source"].iloc[0]}')
print(f'Projection range: {proj["year"].min()}-{proj["year"].max()}')
print(f'Countries covered: {proj["iso_code"].nunique()}')
print()
print('2050 Blue Zone Country Projections (Medium variant):')
for iso, info in BZ_INFO.items():
    p = proj[(proj['iso_code'] == iso) & (proj['year'] == 2050)]
    if len(p) > 0:
        print(f'  {info["name"]}: {p.iloc[0]["le_medium"]:.1f} years')
print()
print('NOTE: These projections are based on trend extrapolation from real historical')
print('data with logistic dampening. They assume no major disruptions (pandemics,')
print('wars) or breakthroughs (medical advances). Use with appropriate caution.')

PROJECTION ANALYSIS SUMMARY

Projection source: trend_extrapolation_from_real_data
Projection range: 2024-2100
Countries covered: 93

2050 Blue Zone Country Projections (Medium variant):
  Japan: 84.5 years
  Italy: 84.2 years
  Greece: 82.0 years
  Costa Rica: 81.1 years
  USA: 78.4 years

NOTE: These projections are based on trend extrapolation from real historical
data with logistic dampening. They assume no major disruptions (pandemics,
wars) or breakthroughs (medical advances). Use with appropriate caution.
